# Coding Project 1: Linear Regression and Regularization

**Please write the names of all group members here:**




---

*Note:* The provided structure for the code below is only suggestive. You may structure your program differently, provided that your notebook is readable, reproducible, and answers all questions clearly.

A minimal `sklearn` workflow example is included before Question 3. It is intended to illustrate the use of `ColumnTransformer`, `Pipeline`, and `GridSearchCV`; it does **not** solve the project questions for you.

### Question 1 - Importing and Preparing the Data

In [3]:
%pip install --upgrade --force-reinstall matplotlib pandas numpy scipy scikit-learn

  Using cached matplotlib-3.11.2-cp312-cp312-macosx_11_0_arm64.whl.metadata (80 kB)
  Using cached pandas-3.0.6-cp312-cp312-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached numpy-2.5.3-cp312-cp312-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached scipy-1.18.1-cp312-cp312-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached scikit_learn-1.9.1-cp312-cp312-macosx_12_0_arm64.whl.metadata (9.3 kB)
  Using cached contourpy-1.4.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (3.8 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.66.1-cp312-cp312-macosx_10_13_universal2.whl.metadata (130 kB)
  Using cached kiwisolver-1.5.1-cp312-cp312-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached packaging-26.3-py3-none-any.whl.metadata (3.5 kB)
  Using cached pillow-12.3.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (9.1 kB)
  Using cached pyparsing-3.3.3-py3-none-any.whl.metadata (5.9 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.

In [4]:
# For Question 1, you may find the following packages useful:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import scipy.stats as stats
import re
from sklearn.model_selection import train_test_split

# 1.a) Import Housing.csv as a pandas DataFrame.
# Separate the explanatory features X from the target y = SalePrice.
#
# Guidance:
# - Refer to data_description.txt to determine how variables should be interpreted.
# - The first column of the csv file is an index and should not be used as an explanatory variable.
# - Keep the original variable names so that later coefficient tables are easy to interpret.

# Variable types come from data_description.txt, not from the dtypes:
# MSSubClass is stored as integers but is categorical.
types = pd.Series(dict(re.findall(r"^(\w+) \((numerical|categorical)\):",
                                  open("data/data_description.txt").read(), flags=re.M)))
NUMERICAL = types.index[types == "numerical"].tolist()
CATEGORICAL = types.index[types == "categorical"].tolist()

# pandas' default reading turns 'NA' and 'None' into NaN, but for many categorical variables
# they are valid levels (e.g. 'NA' = "No Garage"). So only numerical 'NA' is read as missing;
# genuinely missing categorical values are handled in 1.d).
data = pd.read_csv("data/Housing.csv", index_col=0, keep_default_na=False,
                   na_values={col: ["NA"] for col in NUMERICAL},
                   dtype={col: str for col in CATEGORICAL})
X = data.drop(columns="SalePrice")
y = data["SalePrice"]

assert sorted(X.columns) == sorted(types.index)
print(f"X: {X.shape}, y: {y.shape} | numerical: {len(NUMERICAL)}, categorical: {len(CATEGORICAL)}")


# 1.b) Graphically determine whether SalePrice is approximately Gaussian.
# If it is not, suggest and apply a suitable transformation.
# Explain why considering such a transformation can be important.
#
# Guidance:
# - Use suitable graphical diagnostics (for example, a histogram and/or Q-Q plot).
# - Keep track of the transformation you choose because later questions require metrics
#   both on the transformed scale and, after inverse transformation, on the original
#   SalePrice scale.



# 1.c) Split the data into training and test sets.
# Randomly assign 70% of the observations to the training set and 30% to the test set.
#
# IMPORTANT:
# - Save an UNPROCESSED copy of X_train and X_test before doing any imputation,
#   standardization, or one-hot encoding.
# - You will return to these raw splits in Question 3.b) to build a leakage-safe
#   cross-validation pipeline.
#
# Suggested naming:
# X_train_raw, X_test_raw, y_train, y_test



# 1.d) Prepare X_train and X_test using TRAINING-SET information only.
#
# Required steps:
# (i)   Numerical missing values: impute with the corresponding training-set mean.
# (ii)  Categorical missing values: impute with the most frequent category in the
#       corresponding training column.
# (iii) Treat 'NA' / 'None' as actual categorical levels where the data description
#       specifies that they are valid categories, rather than automatically treating
#       them as missing.
# (iv)  Standardize numerical features using training-set means and standard deviations,
#       and apply the same transformation, using these training-set statistics, to
#       the test set.
# (v)   One-hot encode the categorical variables.
# (vi)  Ensure that the test set contains the same encoded feature columns, in the
#       same order, as the training set.
#
# Hint:
# If pd.get_dummies is applied separately to the training and test sets, they may
# produce different dummy-variable columns because some categorical levels occur
# in only one of the two sets. The regression model must receive the same feature
# columns at training and prediction time. One possible way to align the encoded
# test columns to the training columns is:
#
# X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
#
# This creates any training dummy column missing from the test set and fills it
# with zero, while also matching the training-column order.
#
# Expected output / checks:
# - Report the final shapes of the prepared training and test feature matrices.
# - Verify that the encoded feature columns and their order are identical.
# - Do not use any test-set statistic to fit the preprocessing.

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

### Question 2 - Linear Regression on Numerical Features

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import statsmodels.api as sm

# 2.a) Fit an OLS regression using ONLY the numerical predictors and sklearn.
# Include an intercept.
#
# Required output:
# - A table with each numerical feature and its estimated regression coefficient.
# - In-sample and out-of-sample MSE and R^2.
# - If y was transformed in Question 1.b), report MSE and R^2:
#     * on the transformed target scale; and
#     * on the original SalePrice scale after inverse-transforming predictions.
# - Briefly explain what the two scales tell you about model performance.



# 2.b) Reproduce and investigate OLS using matrix algebra on the TRAINING SET ONLY.
#
# Let A denote the design matrix containing the numerical predictors plus a column
# of ones for the intercept.
#
# (i) Compute the estimated coefficients using numpy.
#     Use a numerically stable linear-system solver such as np.linalg.solve(...)
#     whenever the system is numerically well behaved; do not explicitly form
#     a matrix inverse merely to solve the normal equations.
#
#     Expected output:
#     - Estimated intercept and feature coefficients.
#     - A comparison with the sklearn estimates from Question 2.a).



# (ii) Compute the standard error of each estimated coefficient using numpy.
#
#      Expected output:
#      - A table containing coefficient estimates and their standard errors,
#        including the intercept.
#      - Clearly state which estimate corresponds to the intercept.



# (iii) Using your matrix-algebra implementation, verify that the in-sample
#       MSE and R^2 agree with the corresponding results from Question 2.a),
#       up to numerical rounding.



# (iv) Inspect the numerical structure of A.
#
#      Required diagnostics:
#      - rank of A;
#      - singular values of A.
#
#      Guidance:
#      - np.linalg.matrix_rank(...) and np.linalg.svd(..., compute_uv=False)
#        may be useful.
#      - Comment on whether A is full column rank.
#      - Inspect the smallest singular values and discuss whether they suggest
#        potential numerical instability.
#      - Relate this evidence to the OLS calculations above.



# (v) Replace the usual inverse-based expression by the Moore-Penrose pseudoinverse.
#     Use np.linalg.pinv(...) with its default cutoff.
#
#     Required comparison:
#     - Do the coefficient estimates change?
#     - Does the residual variance estimate change?
#     - Do the standard errors change?
#     - Explain when the ordinary inverse-based and pseudoinverse-based results
#       should agree and when they may differ, using the diagnostics from (iv).



# (vi) Confirm your results with statsmodels OLS.
#
#      Required output:
#      - Coefficient estimates and standard errors from statsmodels.
#      - Compare them with your matrix-algebra results up to numerical rounding.
#      - If they differ, relate the discrepancy to the rank / numerical-stability
#        diagnostics above.


### Minimal sklearn Workflow Example (Illustration Only)

In [ ]:
# This example demonstrates the WORKFLOW required in Question 3.b):
#
# raw data
#   -> ColumnTransformer
#   -> Pipeline
#   -> GridSearchCV
#   -> best hyperparameters / CV score
#   -> refitted best estimator
#   -> held-out test evaluation
#
# It uses a small artificial dataset and Ridge regression. It is NOT a solution
# to the housing problem, and its hyperparameter grid should not be copied blindly.

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.metrics import mean_squared_error

# ----- 1. Create a tiny mixed-type example dataset -----
rng = np.random.default_rng(1)
n = 60

X_example = pd.DataFrame({
    "size": rng.normal(100, 15, n),
    "age": rng.normal(20, 7, n),
    "region": rng.choice(["A", "B", "C"], size=n),
})

# Add a few missing values only to illustrate preprocessing.
X_example.loc[[2, 11, 31], "size"] = np.nan
X_example.loc[[5, 27], "region"] = np.nan

# Construct an arbitrary regression target.
region_effect = X_example["region"].map({"A": 0.0, "B": 5.0, "C": -3.0}).fillna(0.0)
y_example = (
    2.0 * X_example["size"].fillna(X_example["size"].mean())
    - 1.5 * X_example["age"]
    + region_effect
    + rng.normal(0, 5, n)
)

X_ex_train, X_ex_test, y_ex_train, y_ex_test = train_test_split(
    X_example, y_example, test_size=0.25, random_state=1
)

# ----- 2. Define preprocessing -----
numeric_features = ["size", "age"]
categorical_features = ["region"]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor_example = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

# ----- 3. Put preprocessing and the model in ONE pipeline -----
pipeline_example = Pipeline([
    ("preprocessor", preprocessor_example),
    ("model", Ridge()),
])

# Pipeline parameter names use step__parameter.
param_grid_example = {
    "model__alpha": [0.1, 1.0, 10.0],
}

# Define the CV folds once. The same idea can be used to ensure fair model comparisons.
cv_example = KFold(n_splits=4, shuffle=True, random_state=1)

search_example = GridSearchCV(
    estimator=pipeline_example,
    param_grid=param_grid_example,
    scoring="neg_mean_squared_error",
    cv=cv_example,
    refit=True,
)

# IMPORTANT:
# During each CV split, the preprocessing steps are fitted only on that fold's
# training observations because they are inside the Pipeline.
search_example.fit(X_ex_train, y_ex_train)

# ----- 4. Inspect the selected configuration and its CV performance -----
best_idx = search_example.best_index_
cv_mean_mse = -search_example.cv_results_["mean_test_score"][best_idx]
cv_std_mse = search_example.cv_results_["std_test_score"][best_idx]

print("Best parameters:", search_example.best_params_)
print(f"Mean CV MSE: {cv_mean_mse:.3f}")
print(f"Std. CV MSE: {cv_std_mse:.3f}")

# ----- 5. Final held-out test evaluation -----
# With refit=True (the default), best_estimator_ has already been refitted on ALL
# of X_ex_train using the selected hyperparameters.
best_model_example = search_example.best_estimator_
test_pred_example = best_model_example.predict(X_ex_test)
test_mse_example = mean_squared_error(y_ex_test, test_pred_example)

print(f"Test MSE: {test_mse_example:.3f}")

# Things to understand before Question 3.b):
# - Why must the preprocessing be inside the Pipeline during CV?
# - Why is the parameter called "model__alpha"?
# - What does refit=True do after CV selects the best hyperparameters?
# - Why should the held-out test set be used only after model selection?

# Visualize the structure of the pipeline
pipeline_example


### Question 3 - Regularization Techniques

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.model_selection import GridSearchCV, KFold

# 3.a) Fit OLS using ALL prepared explanatory variables (numerical + categorical).
#
# Use the prepared Housing dataset from Question 1.d).
#
# Required output:
# - In-sample and out-of-sample MSE and R^2 on the ORIGINAL SalePrice scale.
# - If the target was transformed, inverse-transform predictions before computing
#   these metrics.
# - Compare the results with the regression using only numerical features from
#   Question 2.a).
# - Does including the categorical features improve out-of-sample predictive
#   performance?
# - How does the gap between in-sample and out-of-sample performance change?



# 3.b) Implement and tune:
# - Truncated Pseudoinverse regression;
# - Ridge;
# - Lasso;
# - Elastic Net.
#
# Use 8-fold CV, MSE as the model-selection criterion, and the SAME CV folds
# for all four methods.
#
# IMPORTANT: Start from the UNPROCESSED X_train_raw / X_test_raw saved in 1.c).
# The preprocessing must be INSIDE the Pipeline so that all learned preprocessing
# quantities are estimated separately within each CV training fold.
#
# Follow the workflow below.


# 3.b.i) Build ONE reusable preprocessing component.
#
# It should reproduce the rules from Question 1.d):
# - numerical imputation;
# - numerical standardization;
# - categorical imputation;
# - one-hot encoding;
# - correct treatment of 'NA' / 'None' where these are valid categorical levels.
#
# Make sure a category that is absent from a particular CV training fold does not
# cause an error when its validation fold is transformed.
#
# Hint: OneHotEncoder has an option for handling categories not seen during fit.



# 3.b.ii) Construct one Pipeline for each method using the SAME preprocessor.
#
# Required structures:
#
# Ridge / Lasso / Elastic Net:
# raw data -> preprocessor -> model
#
# Truncated Pseudoinverse:
# raw data -> preprocessor -> TruncatedSVD -> LinearRegression
#
# All regressions should contain an intercept, and the intercept should NOT be
# regularized.
#
# Do not implement a custom TPI estimator: use TruncatedSVD as the dimensionality-
# reduction step before LinearRegression.



# 3.b.iii) Define reasonable hyperparameter grids and tune with GridSearchCV.
#
# Tune:
# - regularization strength for Ridge;
# - regularization strength for Lasso;
# - regularization strength and mixing parameter for Elastic Net;
# - number of retained singular directions for Truncated Pseudoinverse.
#   With sklearn's TruncatedSVD, this is n_components: the number of retained
#   singular components, not a singular-value cutoff.
#
# Pipeline parameters follow the step__parameter convention.
# Examples of the NAMING convention (not recommended parameter values):
# - model__alpha
# - model__l1_ratio
# - svd__n_components
#
# Define ONE 8-fold CV splitter and reuse it for every method so the comparison
# uses exactly the same folds.
#
# Use MSE as the selection criterion. sklearn commonly expresses this as
# scoring="neg_mean_squared_error" because GridSearchCV maximizes scores.



# 3.b.iv) Inspect the selected hyperparameters.
#
# For EACH method:
# - Check whether the selected value lies at or near a boundary of the grid.
# - If it does, extend the relevant range and repeat the search.
# - Briefly justify the final grid / range you report.
#
# Do not stop automatically at the first grid if the optimum appears constrained
# by the range you chose.



# 3.b.v) Evaluate the selected and refitted models.
#
# With refit=True, GridSearchCV refits the selected pipeline on the COMPLETE outer
# training set after cross-validation.
#
# Required final metrics:
# - in-sample MSE and R^2 on the ORIGINAL SalePrice scale;
# - out-of-sample MSE and R^2 on the ORIGINAL SalePrice scale.
#
# If y was transformed, inverse-transform predictions before computing these
# final train/test metrics.
#
# Compare the four regularized methods with BOTH OLS benchmarks from 2.a) and 3.a).



# 3.b) REQUIRED SUMMARY TABLE
#
# Include the four regularized models and the OLS benchmarks from 2.a) and 3.a).
# For quantities that do not apply to OLS, use "--".
#
# Suggested columns:
# Model
# Selected hyperparameter(s)
# Mean CV MSE
# Std. CV MSE
# In-sample MSE
# In-sample R^2
# Out-of-sample MSE
# Out-of-sample R^2
#
# For each regularized model, the CV MSE is the mean validation MSE across the
# 8 folds for the SELECTED hyperparameter configuration.
#
# Hint:
# - search.best_index_ identifies the selected row in search.cv_results_.
# - search.cv_results_["mean_test_score"][search.best_index_] gives the selected
#   mean CV score.
# - search.cv_results_["std_test_score"][search.best_index_] gives its standard
#   deviation across validation folds.
# - If scoring="neg_mean_squared_error", reverse the sign of mean_test_score when
#   reporting MSE. Do NOT reverse the sign of the standard deviation.
#
# If SalePrice was transformed, the CV MSE used for model selection is on the
# transformed target scale. The final in-sample / out-of-sample metrics above
# should be reported on the original SalePrice scale.



# Why is it important that the intercept is not penalized during regularization?



# 3.c) Pipeline parameter notation and refitting.
#
# Explain:
# - What a pipeline parameter name such as model__alpha means.
# - What refit=True in GridSearchCV does after the best hyperparameters have
#   been selected.
# - Why this refitting step is useful before the final evaluation on the
#   held-out test set.



# 3.d) Sparsity versus singular-direction truncation.
#
# Required output / discussion:
# - Number of non-zero FEATURE coefficients for Lasso (exclude the intercept).
# - Number of non-zero FEATURE coefficients for Elastic Net (exclude the intercept).
# - Compare with Ridge, which generally shrinks coefficients without setting them
#   exactly to zero.
# - For Truncated Pseudoinverse, report the SELECTED NUMBER OF SINGULAR DIRECTIONS,
#   rather than counting non-zero regression coefficients.
# - Explain why truncating singular directions is conceptually different from
#   coefficient sparsity in Lasso / Elastic Net.



# 3.e) Final model recommendation.
#
# Which model would you recommend for predicting house prices?
#
# Support the recommendation using BOTH:
# - empirical performance; and
# - the structure of the problem.
#
# Relevant considerations may include:
# - number of features;
# - categorical variables / one-hot encoding;
# - collinearity;
# - sparsity;
# - interpretability;
# - stability / robustness;
# - nonlinearity and other limitations of linear models.
#
# Explain how the strengths and limitations of your selected method align with
# the problem structure.


### Project Report

At the end of the notebook, provide a concise synthesis of your analysis. Do not simply repeat every answer above.

#### Approach
Briefly summarize the preprocessing and modeling workflow.

#### Model selection
Explain how the models and hyperparameters were selected, including the cross-validation procedure.

#### Key results
Summarize the most important quantitative results.

#### Interpretation
Explain what the results imply about the regression methods and the housing problem.

#### Robustness
Discuss relevant checks such as leakage prevention, numerical stability, and hyperparameter-grid adequacy.

#### Limitations
State the main limitations of the analysis and models.